# Spectral Joint Flow (SJFPO) — Online Colab / Kaggle (with internet)

Clone the repo with a GitHub token, install requirements, then run training
via `!python scripts/train.py`. No source is embedded in the notebook.

## 0. Secrets and environment

1. Set `GITHUB_TOKEN` in the environment (Colab secrets or Kaggle secrets).
2. The token is read via `load_secret()` using the runtime environment.
3. Run all cells top to bottom.

In [1]:
# Detect runtime and load secrets from the right store.
import os

OUTPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/working/",
    "local": "./",
}
INPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/input/",
    "local": "./",
}
def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

ENV_NAME = detect_env()
OUTPUT = OUTPUT_DIR[ENV_NAME]

Environment: kaggle


In [2]:
# Clone the repo using the GitHub token from the environment/store.
%cd $OUTPUT
!rm -rf spectral-joint-policy
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
repo_slug = os.getenv("GITHUB_REPO", "dzungphieuluuky/spectral-joint-policy")
repo_url = f"https://github.com/{repo_slug}.git"
if GITHUB_TOKEN:
    repo_url = repo_url.replace("github.com", f"{GITHUB_TOKEN}@github.com")
print(f"Cloning from: {repo_url.replace(GITHUB_TOKEN or '', '<TOKEN>') if GITHUB_TOKEN else repo_url}")
!git clone $repo_url spectral-joint-policy
!git status
# Move into the cloned repo and run directly (no embedded source).
%cd spectral-joint-policy
!pwd

/kaggle/working
Cloning from: https://<TOKEN>@github.com/dzungphieuluuky/spectral-joint-policy.git
Cloning into 'spectral-joint-policy'...
remote: Enumerating objects: 6445, done.
remote: Counting objects: 100% (484/484), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 6445 (delta 333), reused 420 (delta 319), pack-reused 5961 (from 1)
Receiving objects: 100% (6445/6445), 9.13 MiB | 19.60 MiB/s, done.
Resolving deltas: 100% (3149/3149), done.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
/kaggle/working/spectral-joint-policy
/kaggle/working/spectral-joint-policy


In [3]:
%%writefile requirements-colab.txt
# Deep Learning
torch
torchvision
torchaudio
torchcodec
jax
jaxlib
accelerate
diffusers
datasets
huggingface-hub
safetensors
bitsandbytes
einops
einx

# Reinforcement Learning / Robot Learning
gymnasium
gymnasium-robotics
pettingzoo
mujoco
mujoco-mjx
mujoco-warp
skrl
lerobot
jax[cuda12]
flax
jaxlib
optax
orbax-checkpoint

# Scientific / Vision
numpy
scipy
pandas
matplotlib
scikit-learn
scikit-image
opencv-python-headless
pillow
imageio
imageio-ffmpeg
av
moviepy

# Data / Storage
pyarrow

# Experiment Tracking / Config
wandb
tensorboard
draccus
tqdm

Writing requirements-colab.txt


In [4]:
# Detect runtime and choose install source.
if ENV_NAME == "colab":
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:
else:
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:

Using Python 3.12.13 environment at: /usr
Resolved 161 packages in 2.10s
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------     0 B/6.21 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 KiB/6.21 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 KiB/6.21 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 KiB/6.21 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 KiB/6.21 KiB
decorator            ------------------------------ 9.02 KiB/9.02 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 KiB/6.21 KiB
decorator            ------------------------------ 9.02 KiB/9.02 KiB
⠙ Preparing packages... (0/16)
mergedeep            ------------------------------ 6.21 K

In [5]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
if _wandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [6]:
# Run training directly from the cloned repo.
!MPLBACKEND=Agg MUJOCO_GL=egl python scripts/train.py \
    --method sjfpo \
    --env Reacher-v5 \
    --seed 3407 \
    --total_steps 1000000 \
    --eval_steps 100 \
    --save_steps 100000 \
    --config configs/ablations/dct_ppo.yaml

runtime settings: {"compute_dtype": "torch.bfloat16", "dynamo_cache_size_limit": 32, "gpu": {"compute_capability": [7, 5], "device_name": "Tesla T4", "supports": {"bf16": false, "fp16_tensor_cores": true, "tf32": false}}, "storage_dtype": "torch.float32", "tf32_enabled": false, "tf32_matmul_precision": "highest"}
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/draccus/parsers/decoding.py", line 177, in decode_dataclass
    instance = origin(**init_args)
               ^^^^^^^^^^^^^^^^^^^
  File "<string>", line 14, in __init__
  File "<string>", line 76, in __init__
  File "/kaggle/working/spectral-joint-policy/src/spectral_joint_policy/configuration_sjfpo.py", line 116, in __post_init__
    self.validate()
  File "/kaggle/working/spectral-joint-policy/src/spectral_joint_policy/configuration_sjfpo.py", line 288, in validate
    raise ValueError("SJFPO requires 'observation.state' in input_features.")
ValueError: SJFPO requires 'observation.state' in i